# PolyWhisper eval verification (no secrets needed)

All inputs are public: scripts + adapters from `eulogik/polywhisper` on HF, FLEURS audio from `google/fleurs`. Results land in `/kaggle/working` and are pulled via the Kaggle API. Nothing to attach, just run.

Runs: mr prod beam-5 rep1.0; bn/mr clean beam-1 rep1.0 (full); te prod beam-1 rep1.0 on the live test split.

In [ ]:
!pip install -q transformers datasets accelerate peft torch torchaudio soundfile librosa resampy scipy tqdm huggingface_hub

In [ ]:
import shutil
from huggingface_hub import hf_hub_download
REPO = 'eulogik/polywhisper'
for s in ['eval_lang_pure.py', 'train_v3.py', 'normalize_ortho.py']:
    p = hf_hub_download(REPO, s, repo_type='model')
    shutil.copy(p, s)
    print(' ', s)

In [ ]:
import shutil
from huggingface_hub import hf_hub_download
REPO = 'eulogik/polywhisper'
import os
os.makedirs('adapters', exist_ok=True)
ADS = ['polywhisper_output_gpu0/adapters_v3/te_best_prod.pt',
       'polywhisper_output_gpu0/adapters_v3/bn_best_prod.pt',
       'polywhisper_output_gpu1/adapters_v3/mr_best_prod.pt',
       'polywhisper_output_gpu0_clean/adapters_v3/bn_best_clean.pt',
       'polywhisper_output_gpu1_clean/adapters_v3/mr_best_clean.pt']
for a in ADS:
    p = hf_hub_download(REPO, a, repo_type='model')
    shutil.copy(p, 'adapters/' + p.split('/')[-1])
    print(' ', a)

In [ ]:
import json
from pathlib import Path
import numpy as np
import soundfile as sf
from scipy import signal
from datasets import load_dataset
DATA_DIR = Path('polywhisper_output/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
def resample(audio, sr):
    a = np.asarray(audio, dtype=np.float32)
    return a if sr == 16000 else signal.resample(a, int(len(a) * 16000 / sr)).astype(np.float32)
for lang in ['te_in', 'bn_in', 'mr_in']:
    adir = DATA_DIR / f'audio_fleurs_{lang}'
    adir.mkdir(parents=True, exist_ok=True)
    recs = []
    ds = load_dataset('google/fleurs', lang, split='test', streaming=True)
    for item in ds:
        a = item['audio']
        au = resample(a['array'], a['sampling_rate'])
        tx = item['transcription'].strip()
        if len(au) < 1600 or len(au) > 30 * 16000 or len(tx) < 3:
            continue
        wp = adir / f'{len(recs):06d}.wav'
        sf.write(str(wp), au, 16000)
        recs.append({'wav': str(wp), 'text': tx})
    json.dump(recs, open(DATA_DIR / f'fleurs_{lang}_test.json', 'w'))
    print(lang, len(recs), 'records')

In [ ]:
!python eval_lang_pure.py --lang mr --model-size small --adapter mr_best_prod.pt --adapter-dir adapters --test-json polywhisper_output/data/fleurs_mr_in_test.json --out /kaggle/working/eval_mr_prod_beam5_rep10.json --max-new-tokens 256 --encoder-lora --num-beams 5 

In [ ]:
!python eval_lang_pure.py --lang bn --model-size small --adapter bn_best_clean.pt --adapter-dir adapters --test-json polywhisper_output/data/fleurs_bn_in_test.json --out /kaggle/working/eval_bn_clean_beam1_rep10.json --max-new-tokens 256 --encoder-lora --num-beams 1 

In [ ]:
!python eval_lang_pure.py --lang mr --model-size small --adapter mr_best_clean.pt --adapter-dir adapters --test-json polywhisper_output/data/fleurs_mr_in_test.json --out /kaggle/working/eval_mr_clean_beam1_rep10.json --max-new-tokens 256 --encoder-lora --num-beams 1 

In [ ]:
!python eval_lang_pure.py --lang te --model-size small --adapter te_best_prod.pt --adapter-dir adapters --test-json polywhisper_output/data/fleurs_te_in_test.json --out /kaggle/working/eval_te_prod_beam1_rep10.json --max-new-tokens 256 --encoder-lora --num-beams 1 

In [ ]:
import json, glob
for f in sorted(glob.glob('/kaggle/working/eval_*.json')):
    d = json.load(open(f))
    print(f.split('/')[-1], 'raw WER=%.1f' % d['wer'], 'n=%d' % len(d['samples']))
print('ALL EVALS DONE')